# ดึงข้อมูลจริงสำหรับทำอินโฟกราฟิก (Preprocessing ของ HMC)

โน้ตบุ๊กนี้ **จำลองการคัดเลือกช่วง epoch** ทั้งวิธีเดิม (`preprocess_microvolt`: 1 ชุดต่อระยะ เลือก run ที่ยาวที่สุด) และวิธีใหม่ (`preprocess_microvolt_multi5`: เก็บทุกชุด 5 epoch ติดกันที่ไม่ซ้อนกัน)
โดย **อ่านเฉพาะป้ายระยะนอน (sleepscoring) กับ header ของ EDF** ไม่เขียนไฟล์ข้อมูลสัญญาณขนาดใหญ่เลย

ผลลัพธ์ทั้งหมดเขียนลงโฟลเดอร์แยก `dataset/infographic_stats/`

| ไฟล์ | ใช้ทำอะไร |
|---|---|
| `config.json` | ค่าตั้งต้น (256 Hz, 30 วินาที, 7 ช่อง, N_CONSEC ฯลฯ) |
| `subject_stats.csv` | 1 แถวต่อ subject: ระยะเวลา, จำนวน epoch, จำนวนชุดต่อระยะ (ใหม่/เดิม), ครบหรือไม่ |
| `stage_runs.csv` | 1 แถวต่อ (subject, ระยะ): จำนวน epoch, จำนวน run, run ยาวสุด, จำนวนชุด, epoch ที่เหลือทิ้ง |
| `missing_subjects.csv` | **คนที่ไม่ครบ**: ขาดระยะไหน, ระยะที่ขาดมี epoch เท่าไร run ยาวสุดเท่าไร, ระยะอื่นมีกี่ชุด |
| `split_summary.csv` | สรุปต่อ train/val/test |
| `stage_summary.csv` | สรุปต่อระยะ (ทั้งวิธีใหม่และเดิม) |
| `run_length_hist.csv` | การกระจายความยาว run ต่อระยะ |
| `examples.json` | ตัวอย่างจริง: ลำดับป้ายทั้งคืน, ตำแหน่งชุดที่เลือก, ขั้นตอนการสแกนทีละ i |
| `signal_example.npz` | คลื่นจริงรอบ ๆ ชุดตัวอย่าง (สำหรับวาดภาพคลื่นในอินโฟ) |
| `overall_summary.json` | ตัวเลขภาพรวมทั้ง dataset |

In [ ]:
# Section 1 Import และ Path Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install mne -q

import json
import itertools
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

mne.set_log_level("ERROR")

PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset/HMC"
OLD_DIR = PROJECT_DIR / "dataset/preprocess_microvolt"
NEW_DIR = PROJECT_DIR / "dataset/preprocess_microvolt_multi5"
STATS_DIR = PROJECT_DIR / "dataset/infographic_stats"
STATS_DIR.mkdir(parents=True, exist_ok=True)

FS = 256
EPOCH_SEC = 30
EPOCH_LEN = EPOCH_SEC * FS
N_CONSEC = 5
STAGES = ["W", "N1", "N2", "N3", "R"]
SPLITS = ["train", "val", "test"]

CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "chin"]
CHANNEL_NAMES = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin EMG"]
EXCLUDED = ["SN014", "SN064", "SN135"]

# ตั้งชื่อ subject เองได้ ถ้า None โค้ดจะเลือกให้อัตโนมัติ
EXAMPLE_COMPLETE = "SN001"
EXAMPLE_INCOMPLETE = "SN003"
SIGNAL_STAGE = "N2"
EXTRACT_SIGNAL_EXAMPLE = True

In [ ]:
# Section 2 โหลด split เดิม
split_path = OLD_DIR / "split_subjects.json"
if not split_path.exists():
    split_path = NEW_DIR / "split_subjects.json"
assert split_path.exists(), "ไม่พบ split_subjects.json ทั้งใน preprocess_microvolt และ multi5"

with open(split_path) as f:
    split_info = json.load(f)

print("ใช้ split จาก:", split_path)
print("Train:", split_info["n_train"], "| Val:", split_info["n_val"], "| Test:", split_info["n_test"])

In [ ]:
# Section 3 ฟังก์ชัน (ตรรกะเหมือนโค้ดจริงทั้งวิธีเดิมและวิธีใหม่)

def get_labels(subject, n_epochs):
    ann = mne.read_annotations(DATA_DIR / f"{subject}_sleepscoring.edf")
    labels = ["?"] * n_epochs
    onsets = [None] * n_epochs
    for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
        stage = desc.split()[-1]
        if stage not in STAGES:
            continue
        first = int(round(onset / 30))
        last = int(round((onset + duration) / 30))
        for e in range(first, last):
            if e < n_epochs:
                labels[e] = stage
                onsets[e] = onset + (e - first) * 30
    return labels, onsets


def pick_channels(raw):
    picks = []
    for key in CHANNELS:
        found = [i for i, n in enumerate(raw.ch_names) if key.lower() in n.lower()]
        if len(found) != 1:
            return None
        picks.append(found[0])
    return picks


def find_all_runs(labels, stage):
    # วิธีใหม่: เก็บทุกชุด 5 epoch ที่ไม่ซ้อนกัน
    starts = []
    i = 0
    while i + N_CONSEC <= len(labels):
        if all(labels[i + k] == stage for k in range(N_CONSEC)):
            starts.append(i)
            i += N_CONSEC
        else:
            i += 1
    return starts


def find_run_old(labels, stage):
    # วิธีเดิม: run แรกที่ยาวครบ 5 ถ้าไม่มีใช้ run ที่ยาวที่สุด (เท่ากันเลือกอันแรก)
    best_first, best_len = None, 0
    count = 0
    for i, lab in enumerate(labels):
        count = count + 1 if lab == stage else 0
        if count == N_CONSEC:
            return i - N_CONSEC + 1, N_CONSEC
        if count > best_len:
            best_len = count
            best_first = i - count + 1
    return best_first, best_len


def run_lengths(labels, stage):
    # คืน [(epoch เริ่มต้น, ความยาว run)] ของระยะนั้นทั้งคืน
    out, pos = [], 0
    for lab, grp in itertools.groupby(labels):
        n = len(list(grp))
        if lab == stage:
            out.append((pos, n))
        pos += n
    return out


def scan_trace(labels, stage, start=0, max_groups=3, max_steps=300):
    # บันทึกการสแกนวิธีใหม่ทีละ i (ผ่าน -> +5, ไม่ผ่าน -> +1) สำหรับวาดขั้นตอนจริง
    trace, i, found = [], start, 0
    while i + N_CONSEC <= len(labels) and found < max_groups and len(trace) < max_steps:
        window = labels[i:i + N_CONSEC]
        ok = all(l == stage for l in window)
        trace.append({"i": i, "window": window, "ok": ok})
        if ok:
            found += 1
            i += N_CONSEC
        else:
            i += 1
    return trace


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_json_default)

In [ ]:
# Section 4 สแกนทุก subject (อ่านเฉพาะป้าย + header ไม่อ่านสัญญาณ)

all_subjects = [(sp, s) for sp in SPLITS for s in split_info[f"{sp}_subjects"]]

subj_rows, stage_rows = [], []
labels_store, starts_store = {}, {}
run_hist = Counter()

for split, subject in tqdm(all_subjects):
    row = {"split": split, "subject": subject, "status": "ok", "channels_ok": False}

    try:
        raw = mne.io.read_raw_edf(DATA_DIR / f"{subject}.edf", preload=False, verbose="ERROR")
        sfreq = float(raw.info["sfreq"])
        row["sfreq"] = sfreq
        if sfreq != FS:
            row["status"] = f"bad_sampling_rate:{sfreq}"
            subj_rows.append(row)
            continue

        n_times = int(raw.n_times)
        n_epochs = n_times // EPOCH_LEN
        picks = pick_channels(raw)
        row["channels_ok"] = picks is not None
        if picks is None:
            row["status"] = "missing_channels"

        labels, onsets = get_labels(subject, n_epochs)
    except Exception as e:
        row["status"] = f"error:{type(e).__name__}:{e}"
        subj_rows.append(row)
        continue

    row["n_times"] = n_times
    row["duration_h"] = round(n_times / FS / 3600, 3)
    row["n_epochs"] = n_epochs
    row["n_labeled"] = int(sum(l != "?" for l in labels))
    row["n_unknown"] = int(sum(l == "?" for l in labels))
    subj_rows.append(row)

    labels_store[subject] = labels
    starts_store[subject] = {}

    for s in STAGES:
        runs = run_lengths(labels, s)
        lens = [n for _, n in runs]
        starts_new = find_all_runs(labels, s)
        assert len(starts_new) == sum(n // N_CONSEC for n in lens), f"{subject}/{s}: จำนวนชุดไม่ตรงกับสูตร floor(L/5)"
        first_old, n_old = find_run_old(labels, s)
        n_ep = int(sum(lens))

        starts_store[subject][s] = starts_new
        for n in lens:
            run_hist[(s, n)] += 1

        stage_rows.append({
            "split": split,
            "subject": subject,
            "stage": s,
            "n_epochs": n_ep,
            "n_runs": len(lens),
            "longest_run": max(lens, default=0),
            "n_runs_ge5": int(sum(n >= N_CONSEC for n in lens)),
            "groups_new": len(starts_new),
            "epochs_in_groups": len(starts_new) * N_CONSEC,
            "leftover_epochs": n_ep - len(starts_new) * N_CONSEC,
            "epochs_in_short_runs": int(sum(n for n in lens if n < N_CONSEC)),
            "new_first_epoch": starts_new[0] if starts_new else None,
            "old_n_valid": int(n_old) if first_old is not None else 0,
            "old_first_epoch": first_old,
        })

errs = [(r["subject"], r["status"]) for r in subj_rows if r["status"].startswith("error")]
if errs:
    print("subject ที่อ่านไม่ได้:", errs[:5])
assert stage_rows, "ไม่มี subject ไหนอ่านได้เลย ตรวจ DATA_DIR และชื่อไฟล์ sleepscoring"

print("สแกนเสร็จ:", len(subj_rows), "subjects")
print(pd.Series([r["status"].split(":")[0] for r in subj_rows]).value_counts().to_string())

In [ ]:
# Section 5 รวมเป็นตาราง

stage_df = pd.DataFrame(stage_rows)
subj_df = pd.DataFrame(subj_rows)

def wide(col):
    w = stage_df.pivot(index="subject", columns="stage", values=col)[STAGES]
    w.columns = [f"{s}_{col}" for s in STAGES]
    return w

wide_df = pd.concat([wide(c) for c in ["n_epochs", "groups_new", "longest_run", "old_n_valid"]], axis=1).reset_index()
wide_df = wide_df.rename(columns={f"{s}_n_epochs": f"{s}_epochs" for s in STAGES})
subj_df = subj_df.merge(wide_df, on="subject", how="left")

G_NEW = [f"{s}_groups_new" for s in STAGES]
G_OLD = [f"{s}_old_n_valid" for s in STAGES]
subj_df[G_NEW + G_OLD] = subj_df[G_NEW + G_OLD].fillna(0).astype(int)

ok_mask = subj_df["status"].isin(["ok", "missing_channels"])

def missing_stages(r, cols):
    if r["status"] not in ("ok", "missing_channels"):
        return "error"
    if not r["channels_ok"]:
        return "missing channels"
    return ",".join(s for s, c in zip(STAGES, cols) if r[c] == 0)

subj_df["missing_stages_new"] = subj_df.apply(lambda r: missing_stages(r, G_NEW), axis=1)
subj_df["missing_stages_old"] = subj_df.apply(lambda r: missing_stages(r, G_OLD), axis=1)
subj_df["complete_new"] = ok_mask & subj_df["channels_ok"] & (subj_df[G_NEW] > 0).all(axis=1)
subj_df["complete_old"] = ok_mask & subj_df["channels_ok"] & (subj_df[G_OLD] > 0).all(axis=1)

# ตัวเลขที่ "บันทึกจริง": subject ที่ช่องไม่ครบไม่ถูกบันทึกเลยทั้งสองวิธี
saved = subj_df["channels_ok"] & ok_mask
subj_df["groups_total_new"] = subj_df[G_NEW].sum(axis=1)
subj_df["epochs_new"] = subj_df["groups_total_new"] * N_CONSEC
subj_df["groups_total_old"] = (subj_df[G_OLD] > 0).sum(axis=1)
subj_df["epochs_old"] = subj_df[G_OLD].sum(axis=1)
subj_df["old_short_stages"] = subj_df.apply(
    lambda r: ",".join(f"{s}:{r[c]}" for s, c in zip(STAGES, G_OLD) if 0 < r[c] < N_CONSEC), axis=1)
subj_df.loc[~saved, ["groups_total_new", "epochs_new", "groups_total_old", "epochs_old"]] = 0

subj_df.to_csv(STATS_DIR / "subject_stats.csv", index=False, encoding="utf-8-sig")
stage_df.to_csv(STATS_DIR / "stage_runs.csv", index=False, encoding="utf-8-sig")

print("เก็บ (ครบ 5 ระยะ) วิธีใหม่:", int(subj_df["complete_new"].sum()), "/", len(subj_df))
print("ครบ 5 ระยะ วิธีเดิม     :", int(subj_df["complete_old"].sum()), "/", len(subj_df))
display(subj_df.groupby("split")[["complete_new", "complete_old"]].sum())

In [ ]:
# Section 6 คนที่ไม่ครบ: ขาดระยะไหน และระยะอื่นมีกี่ชุด

inc = subj_df[~subj_df["complete_new"]].copy()

def explain(r):
    if r["status"] == "missing_channels":
        return "ช่องสัญญาณไม่ครบ (ไม่ถูกบันทึกเลย)"
    if r["status"] not in ("ok",):
        return r["status"]
    parts = []
    for s in STAGES:
        if r[f"{s}_groups_new"] == 0:
            ep, lg = int(r[f"{s}_epochs"]), int(r[f"{s}_longest_run"])
            parts.append(f"{s}: ไม่มีเลย" if ep == 0 else f"{s}: มี {ep} epoch แต่ run ยาวสุด {lg}")
    return " | ".join(parts)

inc["why_missing"] = inc.apply(explain, axis=1)

cols = (["split", "subject", "status", "missing_stages_new", "why_missing"]
        + G_NEW + ["groups_total_new"] + [f"{s}_epochs" for s in STAGES])
missing_df = inc[cols].sort_values(["split", "subject"]).reset_index(drop=True)
missing_df.to_csv(STATS_DIR / "missing_subjects.csv", index=False, encoding="utf-8-sig")

print("subject ที่ถูกตัดออก:", len(missing_df))
display(missing_df)

# ระยะไหนเป็นตัวทำให้ถูกตัดบ่อยที่สุด
miss_counter = Counter()
for v in missing_df["missing_stages_new"]:
    for s in str(v).split(","):
        if s in STAGES:
            miss_counter[s] += 1
print("\nจำนวน subject ที่ขาดแต่ละระยะ (1 คนขาดได้หลายระยะ):")
print({s: miss_counter.get(s, 0) for s in STAGES})

# ในคนที่ถูกตัด แยกว่า "ไม่มีระยะนั้นเลย" กับ "มีแต่ run สั้นเกินไป"
reason_rows = []
for _, r in missing_df[missing_df["status"] == "ok"].iterrows():
    for s in STAGES:
        if r[f"{s}_groups_new"] == 0:
            reason_rows.append({"stage": s, "absent": r[f"{s}_epochs"] == 0, "short_only": r[f"{s}_epochs"] > 0})
if reason_rows:
    print("\nสาเหตุที่ขาด (นับต่อระยะ):")
    display(pd.DataFrame(reason_rows).groupby("stage")[["absent", "short_only"]].sum().reindex(STAGES).fillna(0).astype(int))

In [ ]:
# Section 7 สรุปต่อ split และต่อระยะ (ทั้งวิธีใหม่และวิธีเดิม)

split_rows = []
for sp in SPLITS:
    d = subj_df[subj_df["split"] == sp]
    k = d[d["complete_new"]]
    r = {
        "split": sp,
        "subjects": len(d),
        "kept_new": len(k),
        "dropped_new": len(d) - len(k),
        "kept_pct": round(100 * len(k) / max(len(d), 1), 1),
        "complete_old": int(d["complete_old"].sum()),
    }
    for s in STAGES:
        r[f"{s}_groups_kept"] = int(k[f"{s}_groups_new"].sum())
    r["groups_kept"] = int(k["groups_total_new"].sum())
    r["epochs_kept"] = r["groups_kept"] * N_CONSEC
    r["groups_old_all_saved"] = int(d["groups_total_old"].sum())
    r["epochs_old_all_saved"] = int(d["epochs_old"].sum())
    r["median_groups_per_subject"] = float(k["groups_total_new"].median()) if len(k) else None
    r["min_groups_per_subject"] = int(k["groups_total_new"].min()) if len(k) else None
    r["max_groups_per_subject"] = int(k["groups_total_new"].max()) if len(k) else None
    split_rows.append(r)

split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(STATS_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")
print("สรุปต่อ split")
display(split_summary)

valid_stage = stage_df[stage_df["subject"].isin(subj_df.loc[saved, "subject"])]
stage_summary = valid_stage.groupby("stage").agg(
    subjects_with_stage=("n_epochs", lambda x: int((x > 0).sum())),
    subjects_with_ge5_run=("groups_new", lambda x: int((x > 0).sum())),
    total_epochs=("n_epochs", "sum"),
    n_runs=("n_runs", "sum"),
    groups_new=("groups_new", "sum"),
    epochs_in_groups=("epochs_in_groups", "sum"),
    leftover_epochs=("leftover_epochs", "sum"),
    epochs_in_short_runs=("epochs_in_short_runs", "sum"),
    old_groups=("old_n_valid", lambda x: int((x > 0).sum())),
    old_full5=("old_n_valid", lambda x: int((x == N_CONSEC).sum())),
    old_epochs=("old_n_valid", "sum"),
).reindex(STAGES)
stage_summary["pct_epochs_used_new"] = (100 * stage_summary["epochs_in_groups"] / stage_summary["total_epochs"]).round(1)
stage_summary.to_csv(STATS_DIR / "stage_summary.csv", encoding="utf-8-sig")
print("\nสรุปต่อระยะ (เฉพาะ subject ที่ช่องสัญญาณครบ)")
display(stage_summary)

In [ ]:
# Section 8 การกระจายความยาว run และสรุปภาพรวม

hist_df = pd.DataFrame(
    [{"stage": s, "run_length": n, "n_runs": c} for (s, n), c in run_hist.items()]
).sort_values(["stage", "run_length"]).reset_index(drop=True)
hist_df.to_csv(STATS_DIR / "run_length_hist.csv", index=False, encoding="utf-8-sig")

lab_all = [l for sid, ls in labels_store.items() if sid in set(subj_df.loc[saved, "subject"]) for l in ls]
overall = {
    "subjects_total": int(len(subj_df)),
    "subjects_kept_new": int(subj_df["complete_new"].sum()),
    "subjects_dropped_new": int((~subj_df["complete_new"]).sum()),
    "subjects_complete_old": int(subj_df["complete_old"].sum()),
    "groups_new_kept": int(subj_df.loc[subj_df["complete_new"], "groups_total_new"].sum()),
    "epochs_new_kept": int(subj_df.loc[subj_df["complete_new"], "epochs_new"].sum()),
    "groups_old_saved": int(subj_df["groups_total_old"].sum()),
    "epochs_old_saved": int(subj_df["epochs_old"].sum()),
    "epochs_labeled_total": int(sum(l != "?" for l in lab_all)),
    "epochs_unknown_total": int(sum(l == "?" for l in lab_all)),
    "median_duration_h": float(subj_df["duration_h"].median()),
    "median_epochs_per_subject": float(subj_df["n_epochs"].median()),
    "bottleneck_stage_counts_among_kept": None,
}
kept_df = subj_df[subj_df["complete_new"]]
if len(kept_df):
    argmin = kept_df[G_NEW].idxmin(axis=1).str.replace("_groups_new", "", regex=False)
    overall["bottleneck_stage_counts_among_kept"] = argmin.value_counts().reindex(STAGES).fillna(0).astype(int).to_dict()

config = {
    "FS": FS, "EPOCH_SEC": EPOCH_SEC, "EPOCH_LEN": EPOCH_LEN, "N_CONSEC": N_CONSEC,
    "STAGES": STAGES, "CHANNELS": CHANNEL_NAMES, "EXCLUDED": EXCLUDED,
    "split_counts": {sp: len(split_info[f"{sp}_subjects"]) for sp in SPLITS},
    "samples_per_epoch": EPOCH_LEN,
    "samples_per_group": EPOCH_LEN * N_CONSEC,
}
save_json(overall, STATS_DIR / "overall_summary.json")
save_json(config, STATS_DIR / "config.json")
print(json.dumps(overall, ensure_ascii=False, indent=2, default=_json_default))

In [ ]:
# Section 9 ตัวอย่างจริงสำหรับวาดอินโฟ: ลำดับป้ายทั้งคืน + ตำแหน่งชุด + ขั้นตอนสแกน

info = subj_df.set_index("subject")
complete = subj_df[subj_df["complete_new"]]

if EXAMPLE_COMPLETE is None and len(complete):
    # คนที่ N2 มี run ยาว (>=12) เพื่อให้เห็นเศษที่ทิ้ง และจำนวนชุดใกล้ค่ากลาง
    cand = complete[complete["N2_longest_run"] >= 12]
    cand = cand if len(cand) else complete
    med = complete["groups_total_new"].median()
    EXAMPLE_COMPLETE = cand.iloc[(cand["groups_total_new"] - med).abs().argsort().values].iloc[0]["subject"]

if EXAMPLE_INCOMPLETE is None:
    cand = subj_df[(~subj_df["complete_new"]) & (subj_df["status"] == "ok")]
    if len(cand):
        EXAMPLE_INCOMPLETE = cand.sort_values("groups_total_new", ascending=False).iloc[0]["subject"]

print("ตัวอย่างที่ครบ     :", EXAMPLE_COMPLETE)
print("ตัวอย่างที่ไม่ครบ  :", EXAMPLE_INCOMPLETE)

def build_example(subject):
    labels = labels_store[subject]
    sd = stage_df[stage_df["subject"] == subject].set_index("stage")
    ex = {
        "subject": subject,
        "split": info.loc[subject, "split"],
        "n_epochs": len(labels),
        "duration_h": info.loc[subject, "duration_h"],
        "complete_new": bool(info.loc[subject, "complete_new"]),
        "missing_stages_new": info.loc[subject, "missing_stages_new"],
        "labels": labels,
        "new_starts": {s: starts_store[subject][s] for s in STAGES},
        "new_groups": {s: int(sd.loc[s, "groups_new"]) for s in STAGES},
        "stage_epochs": {s: int(sd.loc[s, "n_epochs"]) for s in STAGES},
        "longest_run": {s: int(sd.loc[s, "longest_run"]) for s in STAGES},
        "leftover_epochs": {s: int(sd.loc[s, "leftover_epochs"]) for s in STAGES},
        "old_selection": {s: {"first": sd.loc[s, "old_first_epoch"], "n": int(sd.loc[s, "old_n_valid"])} for s in STAGES},
        "runs": {s: run_lengths(labels, s) for s in STAGES},
        "scan_trace": {},
    }
    for s in STAGES:
        first_occ = next((i for i, l in enumerate(labels) if l == s), None)
        ex["scan_trace"][s] = [] if first_occ is None else scan_trace(labels, s, start=max(0, first_occ - 2))
    return ex

examples = {}
for name, sid in [("complete", EXAMPLE_COMPLETE), ("incomplete", EXAMPLE_INCOMPLETE)]:
    if sid is not None and sid in labels_store:
        examples[name] = build_example(sid)

save_json(examples, STATS_DIR / "examples.json")

for name, ex in examples.items():
    print(f"\n[{name}] {ex['subject']} | {ex['n_epochs']} epochs | ครบ: {ex['complete_new']} | ขาด: {ex['missing_stages_new'] or '-'}")
    print("  ชุดวิธีใหม่ :", ex["new_groups"])
    print("  epoch ต่อระยะ:", ex["stage_epochs"])
    print("  run ยาวสุด  :", ex["longest_run"])
    print("  เศษที่ทิ้ง   :", ex["leftover_epochs"])
    print("  วิธีเดิม     :", {s: v["n"] for s, v in ex["old_selection"].items()})

In [ ]:
# Section 10 คลื่นจริงของชุดตัวอย่าง (อ่านสัญญาณแค่ subject เดียว)

if EXTRACT_SIGNAL_EXAMPLE and "complete" in examples:
    sid = examples["complete"]["subject"]
    starts = examples["complete"]["new_starts"][SIGNAL_STAGE]
    assert starts, f"{sid} ไม่มีชุด {SIGNAL_STAGE}"
    first = int(starts[0])
    n_ep = examples["complete"]["n_epochs"]

    ctx_first = max(0, first - 3)
    ctx_last = min(n_ep, first + N_CONSEC + 3)

    raw = mne.io.read_raw_edf(DATA_DIR / f"{sid}.edf", preload=False, verbose="ERROR")
    picks = pick_channels(raw)
    data = raw.get_data(picks=picks, start=ctx_first * EPOCH_LEN, stop=ctx_last * EPOCH_LEN, units="uV")
    X_ctx = data.T.reshape(ctx_last - ctx_first, EPOCH_LEN, len(picks)).astype(np.float32)

    labels = examples["complete"]["labels"]
    np.savez_compressed(
        STATS_DIR / "signal_example.npz",
        X_ctx=X_ctx,
        epoch_indices=np.arange(ctx_first, ctx_last),
        labels_ctx=np.array(labels[ctx_first:ctx_last]),
        group_first_epoch=first,
        group_n=N_CONSEC,
        stage=SIGNAL_STAGE,
        subject=sid,
        fs=FS,
        channels=np.array(CHANNEL_NAMES),
    )
    print(f"บันทึก signal_example.npz: {sid} | ชุด {SIGNAL_STAGE} เริ่ม epoch {first}")
    print("X_ctx shape:", X_ctx.shape, "(epoch, sample, channel) | ชุดอยู่ที่ index", first - ctx_first, "ถึง", first - ctx_first + N_CONSEC - 1)
else:
    print("ข้ามขั้นตอนนี้")

In [ ]:
# Section 11 ตรวจว่าที่จำลองตรงกับไฟล์จริงที่เคยสร้างไว้หรือไม่ (ถ้ามีไฟล์)

def check_old():
    if not OLD_DIR.exists():
        print("ไม่พบ", OLD_DIR, "ข้ามการตรวจวิธีเดิม")
        return
    bad, n_files = [], 0
    for sp in SPLITS:
        for f in sorted((OLD_DIR / sp).glob("SN*.npz")):
            n_files += 1
            with np.load(f, allow_pickle=True) as d:
                actual = {str(s): int(n) for s, n in zip(d["stage"], d["n_valid"])}
            r = info.loc[f.stem]
            sim = {s: int(r[f"{s}_old_n_valid"]) for s in STAGES if r[f"{s}_old_n_valid"] > 0}
            if actual != sim:
                bad.append((f.stem, actual, sim))
    expected_files = int((saved & (subj_df[G_OLD].sum(axis=1) > 0)).sum())
    print(f"วิธีเดิม: ไฟล์จริง {n_files} | คาดว่าต้องมี {expected_files} | ไม่ตรงกัน {len(bad)}")
    for b in bad[:10]:
        print("  ", b)

def check_new():
    if not NEW_DIR.exists():
        print("ไม่พบ", NEW_DIR, "ข้ามการตรวจวิธีใหม่")
        return
    bad, n_files = [], 0
    present = set()
    for sp in SPLITS:
        for f in sorted((NEW_DIR / sp).glob("SN*.npz")):
            n_files += 1
            present.add(f.stem)
            with np.load(f, allow_pickle=True) as d:
                actual = dict(Counter(d["stage"].astype(str)))
            r = info.loc[f.stem]
            sim = {s: int(r[f"{s}_groups_new"]) for s in STAGES if r[f"{s}_groups_new"] > 0}
            if actual != sim:
                bad.append((f.stem, actual, sim))
    expected = set(subj_df.loc[subj_df["complete_new"], "subject"])
    print(f"วิธีใหม่: ไฟล์จริง {n_files} | คาดว่าต้องมี {len(expected)} | จำนวนชุดไม่ตรง {len(bad)}")
    print("  ไฟล์ที่คาดว่ามีแต่ไม่พบ:", sorted(expected - present)[:10])
    print("  ไฟล์ที่พบแต่ไม่ควรมี  :", sorted(present - expected)[:10])
    for b in bad[:10]:
        print("  ", b)

check_old()
check_new()

In [ ]:
# Section 12 ภาพตัวอย่างเพื่อเช็กตัวเลขเบื้องต้น (ยังไม่ใช่อินโฟจริง)

colors = {"W": "#EF9F27", "N1": "#D4537E", "N2": "#1D9E75", "N3": "#7F77DD", "R": "#D85A30"}

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

bottom = np.zeros(len(SPLITS))
for s in STAGES:
    vals = split_summary[f"{s}_groups_kept"].values
    ax[0].bar(SPLITS, vals, bottom=bottom, color=colors[s], label=s)
    bottom += vals
ax[0].set_title("5-epoch groups kept (new method) per split")
ax[0].legend()

ax[1].bar(STAGES, [miss_counter.get(s, 0) for s in STAGES], color=[colors[s] for s in STAGES])
ax[1].set_title("Subjects missing each stage (new method)")

for s in STAGES:
    h = hist_df[(hist_df["stage"] == s) & (hist_df["run_length"] <= 40)]
    ax[2].plot(h["run_length"], h["n_runs"], label=s, color=colors[s])
ax[2].axvline(N_CONSEC, color="gray", linestyle="--")
ax[2].set_yscale("log")
ax[2].set_title("Run-length distribution (dashed = 5)")
ax[2].legend()
plt.tight_layout()
plt.show()

if "complete" in examples:
    ex = examples["complete"]
    code_map = {s: i for i, s in enumerate(["?"] + STAGES)}
    plt.figure(figsize=(16, 2.5))
    plt.step(range(len(ex["labels"])), [code_map[l] for l in ex["labels"]], where="post", linewidth=0.8)
    for s in STAGES:
        for st in ex["new_starts"][s]:
            plt.axvspan(st, st + N_CONSEC, color=colors[s], alpha=0.4)
    plt.yticks(range(len(code_map)), list(code_map))
    plt.title(f"{ex['subject']}: hypnogram and selected groups (colored bands)")
    plt.xlabel("epoch")
    plt.show()

In [ ]:
# Section 13 รายการไฟล์ที่ได้

for p in sorted(STATS_DIR.glob("*")):
    print(f"{p.name:28s} {p.stat().st_size / 1024:10.1f} KB")
print("\nโฟลเดอร์:", STATS_DIR)